In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import rankdata
import pickle
import re

## 1.读取单个股票return

In [ ]:
stock_returns = pd.read_csv('data/raw/kjtt6tbmn0gz9chy.csv')

In [ ]:
stock_returns.head()

return清洗

In [ ]:
# Replace 'B' and 'C' with NaN in the 'RET' column without inplace
stock_returns['RET'] = stock_returns['RET'].replace(['B', 'C'], np.nan)

# Convert the 'RET' column to float
stock_returns['RET'] = stock_returns['RET'].astype(float)

## 2.读取94个因子

In [ ]:
stock_characteristics = pd.read_csv('data/raw/datashare.csv')

In [ ]:
stock_characteristics.head()

In [ ]:
stock_characteristics.columns

### merge 因子（月度 数据）和return

In [ ]:
# merge data
characteristics_with_return = pd.merge(stock_characteristics, stock_returns, \
                                       left_on=['permno', 'DATE'], right_on=['PERMNO', 'date'], how='left')

In [ ]:
characteristics_with_return.head()

In [ ]:
# Delete the row where the return is nan
characteristics_with_return.dropna(subset=['RET'], inplace=True)
characteristics_with_return = characteristics_with_return.drop(columns=['PERMNO', 'date'])

In [ ]:
characteristics_with_return.head()

## 3.宏观指标

In [ ]:
macroeconomic_predictors = pd.read_csv('data/raw/PredictorData2023.xlsx - Monthly.csv', thousands=",")
macroeconomic_predictors.head()

In [ ]:
macroeconomic_predictors['dp'] = np.log(macroeconomic_predictors['D12']) - np.log(macroeconomic_predictors['Index'])
macroeconomic_predictors['EP'] = np.log(macroeconomic_predictors['E12']) - np.log(macroeconomic_predictors['Index'])
macroeconomic_predictors['tms'] = macroeconomic_predictors['lty'] - macroeconomic_predictors['tbl']
macroeconomic_predictors['dfy'] = macroeconomic_predictors['BAA'] - macroeconomic_predictors['AAA']

macroeconomic_predictors = macroeconomic_predictors[['yyyymm', 'dp', 'EP', 'b/m', 'ntis', 'tbl', 'tms', 'dfy', 'svar']]

characteristics_with_return['DATE'] = characteristics_with_return['DATE'].astype(str).str[:6].astype(int)

data = pd.merge(characteristics_with_return, macroeconomic_predictors, left_on=['DATE'], right_on=['yyyymm'], how='left')

In [ ]:
# 计算超额收益
data['real_RET'] = data['RET'] - data['tbl']

In [ ]:
macroeconomic_columns = ['dp', 'EP', 'b/m', 'ntis', 'tbl', 'tms', 'dfy', 'svar']
data.drop(columns=['yyyymm', 'sic2','RET'] + macroeconomic_columns, inplace=True)

### 空值处理

In [ ]:
import pandas as pd
from tqdm import tqdm

def clean_and_fill_na(df):
    """
    过滤掉空值比例大于50%的列，并按照'permno'和'DATE'分组，每个公司最多连续向前填充12个空值。

    参数:
        df (pd.DataFrame): 输入的DataFrame，必须包含'permno'和'DATE'列。

    返回:
        pd.DataFrame: 处理后的DataFrame。
    """
    # 统计每个列空值的比例
    nan_summary = df.isna().sum()
    nan_columns = nan_summary[nan_summary > 0]
    nan_info = pd.DataFrame({
        'Column': nan_columns.index,
        'NaN Count': nan_columns.values,
        'NaN Percentage': (nan_columns.values / len(df)) * 100
    })

    # 筛选出空值比例小于等于50%的列
    columns_to_keep = nan_info[nan_info['NaN Percentage'] <= 50]['Column'].tolist()
    df_filtered = df[columns_to_keep + ['permno', 'DATE','real_RET']]

    # 按照'permno'和'DATE'排序
    df_filtered = df_filtered.sort_values(by=['permno', 'DATE'])

    # 定义填充空值的函数
    def fill_na(group):
        # 仅选择非分组列进行填充
        non_group_columns = group.drop(columns=['permno'])
        non_group_columns = non_group_columns.ffill(limit=12)  # 向前填充最多12个空值
        group[non_group_columns.columns] = non_group_columns  # 更新原来的分组数据
        return group

    # 使用 tqdm 包装 groupby 对象，以显示进度条
    tqdm.pandas(desc="Processing Groups")
    df_filled = df_filtered.groupby('permno', group_keys=False).progress_apply(fill_na)

    # 恢复索引
    df_filled = df_filled.reset_index(drop=True)

    return df_filled

In [ ]:
df_filled = clean_and_fill_na(data)

### 标准化

In [ ]:
def Normal(x):
    
    x = rankdata(x, axis=0, nan_policy='omit')
    
    x = ((x-1) / (np.nanmax(x, axis=0) - 1) - 0.5) * 2
    
    return x

In [ ]:
# cross-sectionally rank
feature_set = df_filled.columns[:-3]
df_filled[feature_set] = df_filled.groupby('DATE')[feature_set].transform(lambda x: Normal(x.values))

In [ ]:
# 假设 df_filled 是你的 DataFrame
permno_col = df_filled['permno']

df_filled = df_filled.groupby('permno').transform(lambda x: x.fillna(x.median()))

df_filled['permno'] = permno_col

In [ ]:
# 对所有剩余的空值填充为0
df_filled = df_filled.fillna(0)

In [ ]:
import os
import pandas as pd
from tqdm import tqdm  # 导入 tqdm 用于显示进度条

# 假设 df_filled 是你的 DataFrame
df_filled['DATE'] = pd.to_datetime(df_filled['DATE'], format='%Y%m')  # 确保 DATE 列是日期类型

# 获取保存目录
base_dir = 'data/processed/yearly'
if not os.path.exists(base_dir):
    os.makedirs(base_dir)

# 按年份分组，并使用 tqdm 显示进度条
for year, group in tqdm(df_filled.groupby(df_filled['DATE'].dt.year), desc="Processing Years"):
    # 保存训练label
    label_file_path = os.path.join(base_dir, f"{year}_real_RET.feather")
    group[['real_RET']].reset_index(drop=True).to_feather(label_file_path)
    print(f"Saved label file: {label_file_path}")
    
    # 保存其他数据
    data_file_path = os.path.join(base_dir, f"{year}_df_data.feather")
    group.drop(columns=['real_RET']).reset_index(drop=True).to_feather(data_file_path)
    print(f"Saved data file: {data_file_path}")